# RAG sul fascicolo: vettoriale o ibrida?

Giorno 3, pomeriggio. 50 minuti, ognuno al proprio portatile.

Costruite una ricerca sul fascicolo in tre modi, misurate quale trova il passo giusto, poi fate rispondere un
modello con le fonti. Tutti i dati sono sintetici.

| Parte | Minuti | Che cosa si fa |
|---|---|---|
| 1. L'indice | 10 | Dividere i documenti in passi, calcolare un vettore per passo |
| 2. Tre ricerche | 20 | Parole chiave, vettoriale, ibrida: per ogni domanda, il passo atteso è tra i primi? |
| 3. La risposta con le fonti | 15 | I passi trovati vanno a un modello; si controlla ogni affermazione |
| 4. La tabella | 5 | I vostri numeri, da incollare nel foglio condiviso |

Le celle si eseguono in ordine, con Maiusc+Invio. Dove trovate **Da fare**, cambiate la cella e rieseguitela.

## 1. L'indice

Più sotto, la cella `indice = Indice()` scarica il modello di embedding (1 GB) se non l'avete già
scaricato stamattina: serve la rete, una volta sola.

In [ ]:
from rag import Indice, parole, passi_del_corpus

passi = passi_del_corpus()
print(len(passi), "passi da", len({r.split(":")[0] for r, _ in passi}), "documenti\n")
for riferimento, testo in passi[:4]:
    print(riferimento, "|", testo[:110])

Un passo è una riga di un documento. Il riferimento `DOC-05:8` vuol dire: riga 8 del file di DOC-05. È ciò che
permette di citare la fonte.

Nei verbali le domande e le risposte stanno su righe diverse. Un passo fatto della sola risposta («Sì, ero in
cucina con lei») non dice a che cosa risponde: per questo ogni risposta viaggia con la sua domanda.

In [ ]:
for riferimento, testo in passi:
    if riferimento in ("DOC-02:8", "DOC-05:6", "DOC-05:8", "DOC-52:6"):
        print(riferimento, "|", testo, "\n")

Guardate le righe 6 e 8 di DOC-05: due IBAN che differiscono per due cifre scambiate. E DOC-52: un altro estratto
conto, di un'altra persona, con altri IBAN. Nel corpus ci sono cinque documenti così, di un'altra frode: servono a
vedere se la ricerca li confonde con i nostri. Un passo è una riga sola, e una riga non dice di quale caso parla.

La ricerca per parole chiave lavora sulle parole di ogni passo. Come si dividono conta:

In [ ]:
print(parole("Che cosa registra il sito alle 21:58:12?"))
print(parole('15 198.51.100.23 - - [02/Mar/2025:21:58:12 +0100] "POST /verifica HTTP/1.1" 302'))

L'orario resta intero, e le parole che non distinguono niente («che», «cosa», «il») spariscono.

Adesso l'indice: le parole di ogni passo per la ricerca per parole chiave, e un vettore per passo per quella
vettoriale.

In [ ]:
indice = Indice()
print(indice.vettori.shape, "= passi x numeri per passo")

## 2. Tre ricerche

Una domanda, tre modi di cercare. `parole` è BM25: conta le parole in comune, e pesa di più quelle rare.
`vettori` confronta il significato. `ibrida` somma le due classifiche.

In [ ]:
def mostra(domanda, k=5):
    print(domanda)
    for modo in ("parole", "vettori", "ibrida"):
        print(f"\n  {modo}")
        for riferimento, testo in indice.cerca(domanda, modo, k):
            print(f"    {riferimento:<10} {testo[:95]}")

mostra("Che cosa risulta dal telefono alle 10:41?")

La risposta sta in `DOC-04:11`. Chi la trova? Adesso una domanda che non usa le parole del documento:

In [ ]:
mostra("Il coniuge era in casa durante la chiamata?")

Il verbale dice «moglie» e «telefonata», non «coniuge» e «chiamata»: la risposta è in `DOC-02:8`.

**Da fare.** Scrivete una vostra domanda sul caso e guardate che cosa trovano le tre ricerche.

In [ ]:
mostra("...")  # la vostra domanda

Due domande non sono una misura. Il banco di prova ne ha quindici, ciascuna con il passo in cui sta la risposta:
sei con un identificativo esatto, sei parafrasi, due in cui due documenti si contraddicono, una a cui il fascicolo
non risponde.

In [ ]:
from prova_banco import MODI, tabella

def stampa(k):
    esiti, conteggi = tabella(indice, k)
    print(f"Primi {k} risultati\n n  tipo             parole  vettori  ibrida  domanda")
    for e in esiti:
        if e["passi_attesi"]:
            print(f"{e['n']:>2}  {e['tipo']:<16} " + "  ".join(f"{'sì' if e[m] else 'no':<6}" for m in MODI) + f"  {e['domanda'][:60]}")
    print("\ntipo             domande  parole  vettori  ibrida")
    for tipo, c in conteggi.items():
        print(f"{tipo:<16} {c['domande']:>7}  {c['parole']:>6}  {c['vettori']:>7}  {c['ibrida']:>6}")
    return conteggi

conteggi = stampa(5)

Leggete la tabella per tipo di domanda, non il totale.

- Sugli identificativi chi vince? Guardate la domanda 2: perché la ricerca vettoriale non trova un orario?
- Sulle parafrasi chi vince? Guardate la domanda 10: la chat è in inglese, la domanda in italiano.
- Le contraddizioni chiedono due passi insieme, uno per versione. Chi li riporta tutti e due?
- Le domande 8 e 12 non le trova nessuno. Aprite la 12 con `mostra(...)`: che cosa trova al suo posto?

**Da fare.** Rifate la tabella con i primi 3 risultati. Cambia qualcosa? Che cosa vi dice su dove sta il passo
giusto, quando una ricerca lo trova?

In [ ]:
conteggi_3 = stampa(3)

## 3. La risposta con le fonti

I primi cinque passi della ricerca ibrida vanno a un modello, con un'istruzione: rispondere solo con quei passi,
citare la fonte di ogni affermazione, dire quando il fascicolo non basta.

Il modello è quello scritto in `LAB_MODEL` nel file `.env` di questa cartella. Se manca, o se la rete non va, il
notebook usa risposte già calcolate: il resto dell'esercizio non cambia.

In [ ]:
import csv
from risposte import QUI, leggi_env, precalcolata, prompt, rispondi

leggi_env()
with (QUI / "banco.csv").open(encoding="utf-8") as file:
    banco = {d["n"]: d for d in csv.DictReader(file)}

print(prompt(banco["1"]["domanda"], indice.cerca(banco["1"]["domanda"], "ibrida", 5)))

Questo è tutto ciò che il modello vede. Adesso la risposta:

In [ ]:
async def chiedi(numero, tutto=False):
    domanda = banco[numero]["domanda"]
    passi_per_il_modello = indice.passi if tutto else indice.cerca(domanda, "ibrida", 5)
    try:
        risposta, origine = await rispondi(domanda, passi_per_il_modello), "Risposta del modello, adesso."
    except Exception as errore:  # in aula l'esercizio continua anche senza modello
        risposta, quando = precalcolata(numero, "tutto" if tutto else "rag")
        origine = f"Il modello non ha risposto: {str(errore)[:200]}\nQuesta è una risposta già pronta, {quando}."
    print(f"Domanda {numero}: {domanda}\n\n[{origine}]\n\n{risposta}\n")
    if not tutto:
        print("Passi dati al modello:", ", ".join(r for r, _ in passi_per_il_modello))
    print("Passi attesi:", banco[numero]["passi_attesi"] or "nessuno: il fascicolo non lo dice")

await chiedi("1")

**Da fare.** Per ogni affermazione della risposta, aprite il passo citato (la cella qui sotto) e controllate che
lo dica davvero. È la regola del corso: un risultato è un indizio, si controlla sulla fonte.

In [ ]:
def apri(riferimento):
    print(riferimento, "|", dict(indice.passi)[riferimento])

apri("DOC-05:8")

Una contraddizione: due documenti danno due importi diversi per lo stesso bonifico. Il modello li riporta tutti e due?
E gli importi che cita sono tutti del nostro caso? Aprite ogni fonte con `apri(...)` e guardate di chi parla il documento.

In [ ]:
await chiedi("13")

La domanda a cui il fascicolo non risponde. Che cosa fa il modello? Un orario, qui, sarebbe inventato: tra i passi
non ce n'è nessuno. Guardate però quali documenti cita: se compare una data di febbraio, viene dall'altra frode. Una
fonte vera, del caso sbagliato.

In [ ]:
await chiedi("15")

La domanda 12, una delle due che nessuna ricerca trova. Il modello riceve cinque passi sbagliati: che cosa risponde?

In [ ]:
await chiedi("12")

Il confronto: gli stessi quesiti con **tutto il fascicolo** nel prompt, senza ricerca. Con quindici documenti ci
sta. Risponde meglio? E con quarantamila documenti?

In [ ]:
await chiedi("12", tutto=True)

In [ ]:
await chiedi("13", tutto=True)

## 4. La tabella

Incollate questa riga nel foglio condiviso: la vediamo insieme sulla prossima slide.

In [ ]:
righe = []
for nome, c in (("primi 5", conteggi), ("primi 3", conteggi_3)):
    for tipo, valori in c.items():
        righe.append(f"{nome}\t{tipo}\t{valori['domande']}\t{valori['parole']}\t{valori['vettori']}\t{valori['ibrida']}")
print("risultati\ttipo\tdomande\tparole\tvettori\tibrida")
print("\n".join(righe))

Quattro domande per la discussione:

1. Per quale tipo di domanda usereste ciascuna ricerca?
2. Dove la ricerca ibrida ha fatto peggio della migliore delle altre due, e perché?
3. Quando il passo giusto non è tra quelli trovati, che cosa ha fatto il modello?
4. Una citazione che esiste basta? Che cosa manca a un passo per sapere di quale caso parla?